In [1]:
import os
import sys
import json
import yaml
from typing import List
from dotenv import load_dotenv
from pydantic import BaseModel

from agent_framework import Agent, tool
from azure.identity import DefaultAzureCredential
from agent_framework.azure import AzureOpenAIResponsesClient
from agent_framework.devui import serve

# Resolve paths relative to project root
_mqa_dir = os.path.abspath(os.path.join("..", "agents", "mqa-agent"))
_project_root = os.path.abspath("..")

sys.path.insert(0, _project_root)
sys.path.insert(0, _mqa_dir)

from prompt_builder import PromptBuilder

_config_path = os.path.join(_mqa_dir, "config.yaml")
_builder = PromptBuilder(_config_path)

load_dotenv()

True

In [2]:
# ---------------------------------------------------------------------------
# Tool 1: get_available_categories
# ---------------------------------------------------------------------------
@tool
def get_available_categories() -> str:
    """Return the list of available query categories and their associated parameters.

    Call this first to understand which categories can be tagged to a user query.
    Each category has a name and a list of dimension parameters.
    """
    categories = []
    for item in _builder.config.get("categories", []):
        if isinstance(item, dict) and item.get("name"):
            categories.append(
                {"name": item["name"], "parameters": item.get("parameters", [])}
            )
    return json.dumps(categories, indent=2)


# ---------------------------------------------------------------------------
# Tool 2: get_parameters_for_categories
# ---------------------------------------------------------------------------
@tool
def get_parameters_for_categories(categories: List[str]) -> str:
    with open(_config_path, "r") as f:
        config = yaml.safe_load(f)
    parameter_dict = {}
    for category in categories:
        if category not in [c["name"] for c in config["categories"]]:
            continue  # In a real implementation, you might want to handle unknown categories
        for c in config["categories"]:
            if c["name"] == category:
                for param in c["parameters"]:
                    parameter_dict[param] = config["parameters"][param]
    return json.dumps(parameter_dict)

In [4]:
# ---------------------------------------------------------------------------
# Agent instructions
# ---------------------------------------------------------------------------
MQA_INSTRUCTIONS = (
    "You are a Multi-Query Agent designed to help expand user queries into multiple sub-queries based on predefined categories and parameters. "
    "Your goal is to identify relevant categories for a given user query, extract associated parameters, and generate sub-queries that can be used to retrieve data from a database.\n\n"
    "Steps to follow:\n"
    "1. Analyze the user query and determine which categories from the provided list are relevant. You can select multiple categories if applicable.\n"
    "2. For each selected category, identify the associated parameters and their possible values from the configuration.\n"
    "3. Generate multiple sub-queries that combine the user query with the selected categories and parameters. Each sub-query should be a valid question that could be asked to a database or search engine.\n\n"
    "Use the following tools to assist you:\n"
    "- get_available_categories: Returns the list of available categories and their parameters.\n"
    "- get_parameters_for_categories: Given a list of categories, returns the associated parameters and their values.\n\n"
    "Make sure to provide clear and concise sub-queries that cover different aspects of the user's original query based on the selected categories and parameters."
    "Return the shortlisted categories and sub-queries in a structured format as defined by the MQAResponse model."
)

In [5]:
class MQAResponse(BaseModel):
    categories_selected: List[str]
    sub_queries: List[str]


client = AzureOpenAIResponsesClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    deployment_name=os.environ["AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME"],
    credential=DefaultAzureCredential(),
)

agent = Agent(
    name="mqa",
    client=client,
    instructions=MQA_INSTRUCTIONS,
    tools=[get_available_categories, get_parameters_for_categories],
)

In [6]:
import random

def write_categories_jsonl(results: list[dict], filename: str = "predicted_categories.jsonl"):
    """Write query, predicted categories, and ground truth categories to a JSONL file.

    Each line is a JSON object with keys 'query', 'predicted_categories', and 'ground_truth_categories'.
    Overwrites the file if it already exists.
    """
    data_dir = os.path.join(_project_root, "data")
    filepath = os.path.join(data_dir, filename)

    os.makedirs(data_dir, exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        for record in results:
            f.write(json.dumps(record) + "\n")

    print(f"Wrote {len(results)} records to {filepath}")
    return filepath

In [7]:
queries = [
    "How has my drug performed over the last 6 months for Drug D1?",
    "How has my drug performed in 2024 for Drug D1?",
    "How is performance split by shipments, overall patients, and N/R patients for Drug D1?",
    "What has changed over the time period to explain the performance for Drug D1?",
    "For Drug D1, What actions should I take to increase market share, shipments, or patients?",
]

records = []
for i, query in enumerate(queries):
    result = await agent.run(query, options={"response_format": MQAResponse})
    categories = list(map(str, result.value.categories_selected))

    # ground_truth = same as predicted, but drop 1 random category for the 2nd query
    ground_truth = list(categories)
    if i == 1 and len(ground_truth) > 1:
        ground_truth.remove(random.choice(ground_truth))

    print(f"User query: {query}")
    print(f"  predicted:    {categories}")
    print(f"  ground_truth: {ground_truth}")

    records.append({
        "query": query,
        "predicted_categories": categories,
        "ground_truth_categories": ground_truth,
    })

write_categories_jsonl(records)

User query: How has my drug performed over the last 6 months for Drug D1?
  predicted:    ['performance_trends', 'comparative_analysis', 'patient_dynamics']
  ground_truth: ['performance_trends', 'comparative_analysis', 'patient_dynamics']
User query: How has my drug performed in 2024 for Drug D1?
  predicted:    ['performance_trends', 'patient_dynamics', 'prescriber_insights', 'comparative_analysis', 'prescriptive_actions']
  ground_truth: ['performance_trends', 'prescriber_insights', 'comparative_analysis', 'prescriptive_actions']
User query: How is performance split by shipments, overall patients, and N/R patients for Drug D1?
  predicted:    ['performance_trends', 'patient_dynamics', 'operational_friction', 'comparative_analysis', 'prescriptive_actions']
  ground_truth: ['performance_trends', 'patient_dynamics', 'operational_friction', 'comparative_analysis', 'prescriptive_actions']
User query: What has changed over the time period to explain the performance for Drug D1?
  predicte

'c:\\HVE-CORE-V2\\H-star\\H-Star-cleanup\\h-star-custom\\data\\predicted_categories.jsonl'

In [10]:
# python -m src.agent_evaluation.agentic_ops.runner --config_file src/evaluations/offline/agentic_category_evals/experiment.yaml

import subprocess
import sys

!cd "{_project_root}" && "{sys.executable}" -m src.agent_evaluation.agentic_ops.runner --config_file src/evaluations/offline/agentic_category_evals/experiment.yaml


2026-03-12 15:01:18 +0530   46792 execution.bulk     INFO     Finished 5 / 5 lines.
2026-03-12 15:01:18 +0530   46792 execution.bulk     INFO     Average execution time for completed lines: 0.0 seconds. Estimated time for incomplete lines: 0.0 seconds.
======= Run Summary =======

Run name: "custom_agents_category_accuracy_eval_20260312_093118_145288"
Run status: "Completed"
Start time: "2026-03-12 09:31:18.145288+00:00"
Duration: "0:00:01.000982"

2026-03-12 15:01:37 +0530   40380 execution.bulk     INFO     Finished 1 / 5 lines.
2026-03-12 15:01:37 +0530   40380 execution.bulk     INFO     Average execution time for completed lines: 19.05 seconds. Estimated time for incomplete lines: 76.2 seconds.
2026-03-12 15:01:37 +0530   40380 execution.bulk     INFO     Finished 2 / 5 lines.
2026-03-12 15:01:37 +0530   40380 execution.bulk     INFO     Average execution time for completed lines: 9.58 seconds. Estimated time for incomplete lines: 28.74 seconds.
2026-03-12 15:01:37 +0530   40380 e

INFO:__main__:[MAIN] Running step → src.evaluations.offline.agentic_category_evals.evaluator.eval_main.eval_main
INFO:__main__:Calling function eval_main with args
INFO:src.evaluations.offline.agentic_category_evals.evaluator.eval_main:[EVALUATION][EVAL MAIN] - Evaluation begin: input_file_path=c:\HVE-CORE-V2\H-star\H-Star-cleanup\h-star-custom\src/evaluations/offline/agentic_category_evals/datasets/predicted_categories_response.jsonl, results_file_path=c:\HVE-CORE-V2\H-star\H-Star-cleanup\h-star-custom\src/evaluations/offline/agentic_category_evals/report/Agentic_Category_evaluation.json, eval_name=Agentic_Category_evaluation
INFO:src.agent_evaluation.agentic_ops.run_eval:[EVALUATION][LOCAL] - Running evaluation locally without Azure AI project.
INFO:src.evaluations.offline.agentic_category_evals.eval_factory:[EVALUATOR][CONFIG] Successfully retrieved evaluator factory: 'custom_agents_category_evaluator' -> EvaluateAgentsCategories
INFO:src.evaluations.offline.agentic_category_evals.e

In [ ]:
import pandas as pd
from pathlib import Path

file_path = Path(r"C:\HVE-CORE-V2\H-star\H-Star-cleanup\h-star-custom\src\evaluations\offline\agentic_category_evals\report\Agentic_Category_evaluation.json")

# Load JSON
with open(file_path, "r", encoding="utf-8") as f:
    payload = json.load(f)

# Resolve row-wise records from common JSON shapes
rows = None
if isinstance(payload, list):
    rows = payload
elif isinstance(payload, dict):
    for k in ["rows", "results", "records", "data", "evaluations", "items"]:
        if k in payload and isinstance(payload[k], list):
            rows = payload[k]
            break
    if rows is None:
        rows = [payload]
else:
    raise ValueError("Unsupported JSON structure")

df = pd.json_normalize(rows, sep=".")

# Select only the required columns
display_cols = [
    "inputs.query",
    "inputs.response",
    "inputs.predicted_categories",
    "inputs.ground_truth_categories",
    "outputs.custom_agents_category_accuracy_eval.agents_category_accuracy",
    "outputs.custom_agents_category_accuracy_eval.agents_category_recall",
    "outputs.relevance_eval.relevance",
    "outputs.relevance_eval.relevance_reason",
]

display(df[display_cols])

=== Row-wise Table ===


,inputs.query,inputs.response,inputs.predicted_categories,inputs.ground_truth_categories,outputs.custom_agents_category_accuracy_eval.recall@1,outputs.custom_agents_category_accuracy_eval.recall@2,outputs.custom_agents_category_accuracy_eval.recall@3,outputs.custom_agents_category_accuracy_eval.agents_category_accuracy,outputs.custom_agents_category_accuracy_eval.agents_category_recall,outputs.relevance_eval.relevance,...,outputs.relevance_eval.relevance_result,outputs.relevance_eval.relevance_threshold,outputs.relevance_eval.relevance_reason,outputs.relevance_eval.relevance_prompt_tokens,outputs.relevance_eval.relevance_completion_tokens,outputs.relevance_eval.relevance_total_tokens,outputs.relevance_eval.relevance_finish_reason,outputs.relevance_eval.relevance_model,outputs.relevance_eval.relevance_sample_input,outputs.relevance_eval.relevance_sample_output
0,How has my drug performed over the last 6 mont...,Drug D1 shipments declined 8% over the last 6 ...,"[performance_trends, comparative_analysis, pat...","[performance_trends, comparative_analysis, pat...",0.333333,0.666667,1.0,True,1.0,5.0,...,pass,3,The response directly addresses the user's que...,1617,59,1676,stop,gpt-4.1-2025-04-14,"[{""role"": ""user"", ""content"": ""{\""query\"": \""Ho...","[{""role"": ""assistant"", ""content"": ""{\n \""expl..."
1,How has my drug performed in 2024 for Drug D1?,"In 2024, Drug D1 saw a 5% year-over-year incre...","[performance_trends, patient_dynamics, prescri...","[performance_trends, prescriber_insights, comp...",0.250000,0.250000,0.5,False,1.0,5.0,...,pass,3,The response directly addresses Drug D1's perf...,1620,58,1678,stop,gpt-4.1-2025-04-14,"[{""role"": ""user"", ""content"": ""{\""query\"": \""Ho...","[{""role"": ""assistant"", ""content"": ""{\n \""expl..."
2,"How is performance split by shipments, overall...","Drug D1 had 12K shipments, 8.5K overall patien...","[performance_trends, patient_dynamics, operati...","[performance_trends, patient_dynamics, operati...",0.200000,0.400000,0.6,True,1.0,4.0,...,pass,3,The response directly answers the user's query...,1623,60,1683,stop,gpt-4.1-2025-04-14,"[{""role"": ""user"", ""content"": ""{\""query\"": \""Ho...","[{""role"": ""assistant"", ""content"": ""{\n \""expl..."
3,What has changed over the time period to expla...,Key changes include a 15% drop in new prescrib...,"[performance_trends, patient_dynamics, prescri...","[performance_trends, patient_dynamics, prescri...",0.166667,0.333333,0.5,True,1.0,4.0,...,pass,3,The response identifies two specific changes—d...,1610,61,1671,stop,gpt-4.1-2025-04-14,"[{""role"": ""user"", ""content"": ""{\""query\"": \""Wh...","[{""role"": ""assistant"", ""content"": ""{\n \""expl..."
4,"For Drug D1, What actions should I take to inc...","Focus on re-engaging lapsed prescribers, expan...","[performance_trends, patient_dynamics, prescri...","[performance_trends, patient_dynamics, prescri...",0.166667,0.333333,0.5,True,1.0,4.0,...,pass,3,The response directly addresses the user's que...,1620,60,1680,stop,gpt-4.1-2025-04-14,"[{""role"": ""user"", ""content"": ""{\""query\"": \""Fo...","[{""role"": ""assistant"", ""content"": ""{\n \""expl..."



=== Overall Summary ===
Total rows: 5
Total columns: 21
Columns: ['inputs.query', 'inputs.response', 'inputs.predicted_categories', 'inputs.ground_truth_categories', 'outputs.custom_agents_category_accuracy_eval.recall@1', 'outputs.custom_agents_category_accuracy_eval.recall@2', 'outputs.custom_agents_category_accuracy_eval.recall@3', 'outputs.custom_agents_category_accuracy_eval.agents_category_accuracy', 'outputs.custom_agents_category_accuracy_eval.agents_category_recall', 'outputs.relevance_eval.relevance', 'outputs.relevance_eval.gpt_relevance', 'outputs.relevance_eval.relevance_result', 'outputs.relevance_eval.relevance_threshold', 'outputs.relevance_eval.relevance_reason', 'outputs.relevance_eval.relevance_prompt_tokens', 'outputs.relevance_eval.relevance_completion_tokens', 'outputs.relevance_eval.relevance_total_tokens', 'outputs.relevance_eval.relevance_finish_reason', 'outputs.relevance_eval.relevance_model', 'outputs.relevance_eval.relevance_sample_input', 'outputs.relevan

,count,mean,std,min,25%,50%,75%,max
outputs.custom_agents_category_accuracy_eval.recall@1,5.0,0.223333,0.070317,0.166667,0.166667,0.200000,0.25,0.333333
outputs.custom_agents_category_accuracy_eval.recall@2,5.0,0.396667,0.160035,0.250000,0.333333,0.333333,0.40,0.666667
outputs.custom_agents_category_accuracy_eval.recall@3,5.0,0.620000,0.216795,0.500000,0.500000,0.500000,0.60,1.000000
outputs.custom_agents_category_accuracy_eval.agents_category_recall,5.0,1.000000,0.000000,1.000000,1.000000,1.000000,1.00,1.000000
outputs.relevance_eval.relevance,5.0,4.400000,0.547723,4.000000,4.000000,4.000000,5.00,5.000000
outputs.relevance_eval.gpt_relevance,5.0,4.400000,0.547723,4.000000,4.000000,4.000000,5.00,5.000000
outputs.relevance_eval.relevance_threshold,5.0,3.000000,0.000000,3.000000,3.000000,3.000000,3.00,3.000000
outputs.relevance_eval.relevance_prompt_tokens,5.0,1618.000000,4.949747,1610.000000,1617.000000,1620.000000,1620.00,1623.000000
outputs.relevance_eval.relevance_completion_tokens,5.0,59.600000,1.140175,58.000000,59.000000,60.000000,60.00,61.000000
outputs.relevance_eval.relevance_total_tokens,5.0,1677.600000,4.505552,1671.000000,1676.000000,1678.000000,1680.00,1683.000000



Categorical summary (top 3 values per column):

inputs.query:
inputs.query
How has my drug performed over the last 6 months for Drug D1?                             1
How has my drug performed in 2024 for Drug D1?                                            1
How is performance split by shipments, overall patients, and N/R patients for Drug D1?    1

inputs.response:
inputs.response
Drug D1 shipments declined 8% over the last 6 months, with a notable dip in Q3 driven by lower new patient starts.               1
In 2024, Drug D1 saw a 5% year-over-year increase in total patients but a 3% drop in market share due to competitor launches.    1
Drug D1 had 12K shipments, 8.5K overall patients, and 2.1K new/restarted patients in the reporting period.                       1

inputs.predicted_categories:
inputs.predicted_categories
['performance_trends', 'patient_dynamics', 'prescriber_insights', 'operational_friction', 'comparative_analysis', 'prescriptive_actions']    2
['performance_trends